<a href="https://colab.research.google.com/github/Realbenjizo/python-functions-for-data-engineers/blob/main/Chapter_10_Using_Async_to_Scale_Data_Pipelines.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import logging

for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)

In [ ]:
import pandas as pd

# Replace with link to your downloaded file
transactions = "/content/drive/MyDrive/data_files/transactions.csv"

transactions_df = pd.read_csv(transactions)
print(transactions_df)

   user_id  amount transaction_date     status
0      1.0  250.50        1 03 2025  completed
1      2.0    0.00        2 03 2025    pending
2      1.0  175.75        1 03 2025  completed
3      3.0   89.99        3 03 2025  completed
4      2.0  -10.00        2 03 2025     failed
5      4.0  320.00        4 03 2025  completed
6      NaN  150.00        1 03 2025  completed
7      1.0  200.00        5 03 2025  completed


In [ ]:
import asyncio

async def greeting():
    print("Hello my friends")

async def main():
    print("Starting a new task...")

    # Schedule a task that waits for 2 seconds
    sleep_task = asyncio.create_task(asyncio.sleep(2))

    # Wait for the sleep task to finish
    await sleep_task

    # Schedule the greeting coroutine
    greeting_task = asyncio.create_task(greeting())

    # Wait for the greeting to finish
    await greeting_task

    print("The task is finished!")

await main()

Starting a new task...
Hello my friends
The task is finished!


In [ ]:
coroutine=greeting()
print(coroutine)

<coroutine object greeting at 0x7d58871db580>


In [ ]:
# Schedule a task that waits for 2 seconds
sleep_task = asyncio.create_task(asyncio.sleep(2))
# Schedule the greeting coroutine
greeting_task = asyncio.create_task(greeting())

## **Basic Pattern with httpx**

In [ ]:
import asyncio
import httpx
from typing import List, Dict

async def fetch_user(client: httpx.AsyncClient, user_id: int) -> Dict:
    """Single async API call."""
    resp = await client.get(f"https://jsonplaceholder.typicode.com/users/{user_id}")
    resp.raise_for_status()
    return resp.json()

async def fetch_all_users(user_ids: List[int]) -> List[Dict]:
    """Fetch many users concurrently."""
    async with httpx.AsyncClient(timeout=10.0) as client:
        tasks = [fetch_user(client, uid) for uid in user_ids]
        return await asyncio.gather(*tasks, return_exceptions=True)

users = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

results = await fetch_all_users(users)
results

Hello my friends


[{'id': 1,
  'name': 'Leanne Graham',
  'username': 'Bret',
  'email': 'Sincere@april.biz',
  'address': {'street': 'Kulas Light',
   'suite': 'Apt. 556',
   'city': 'Gwenborough',
   'zipcode': '92998-3874',
   'geo': {'lat': '-37.3159', 'lng': '81.1496'}},
  'phone': '1-770-736-8031 x56442',
  'website': 'hildegard.org',
  'company': {'name': 'Romaguera-Crona',
   'catchPhrase': 'Multi-layered client-server neural-net',
   'bs': 'harness real-time e-markets'}},
 {'id': 2,
  'name': 'Ervin Howell',
  'username': 'Antonette',
  'email': 'Shanna@melissa.tv',
  'address': {'street': 'Victor Plains',
   'suite': 'Suite 879',
   'city': 'Wisokyburgh',
   'zipcode': '90566-7771',
   'geo': {'lat': '-43.9509', 'lng': '-34.4618'}},
  'phone': '010-692-6593 x09125',
  'website': 'anastasia.net',
  'company': {'name': 'Deckow-Crist',
   'catchPhrase': 'Proactive didactic contingency',
   'bs': 'synergize scalable supply-chains'}},
 {'id': 3,
  'name': 'Clementine Bauch',
  'username': 'Samantha

In [ ]:
def fetch_users_in_parallel(user_ids: List[int]) -> pd.DataFrame:

    with ThreadPoolExecutor() as executor:
        results = list(executor.map(fetch_user_from_api, user_ids))

    return pd.DataFrame(results)

## **Building Async Pipeline**

In [ ]:
import asyncio
import httpx
from typing import List, Dict
import pandas as pd
import time
import logging

logger = logging.getLogger(__name__)

async def fetch_user(client: httpx.AsyncClient, user_id: int) -> Dict:
    """Single async API call."""
    resp = await client.get( f"https://jsonplaceholder.typicode.com/users/{user_id}")
    resp.raise_for_status()
    data = resp.json()
    return data

async def fetch_all_users(user_ids: List[int]) -> List[Dict]:
    """Fetch many users concurrently."""
    async with httpx.AsyncClient(timeout=10.0) as client:
        tasks = [fetch_user(client, uid) for uid in user_ids]
        result = await asyncio.gather(*tasks, return_exceptions=True)
        return result

async def enrich_with_user_names_async(
    df: pd.DataFrame,
    user_ids: List[int]
) -> pd.DataFrame:
    """Async version of enrichment."""
    df = df.copy()

    user_data = await fetch_all_users(user_ids)
    names = {u['id']: u.get('name', 'Unknown') for u in user_data if isinstance(u, dict)}

    df['user_name'] = df['user_id'].map(names).fillna('Unknown')
    return df

async def fetch_data_in_async_helper(df:pd.DataFrame, user_ids: list[int]) -> pd.DataFrame:
    """Helper to run parallel waiting and show timing."""
    logger.info("Starting async fetch...")
    start_time = time.time()

    users_df =  await enrich_with_user_names_async(df, user_ids)

    duration = time.time() - start_time
    logger.info(f"Successfully fetched await {len(users_df)} users in {duration:.2f} seconds")
    return users_df

# Fetch data for multiple users
if __name__ == "__main__":
    logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
    )
    user_ids = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
    results = await fetch_data_in_async_helper(transactions_df, user_ids)
    print(results.head())

2026-11-07 17:59:32,472 - INFO - Starting async fetch...
2026-11-07 17:59:32,564 - INFO - HTTP Request: GET https://jsonplaceholder.typicode.com/users/1 "HTTP/1.1 200 OK"
2026-11-07 17:59:32,569 - INFO - HTTP Request: GET https://jsonplaceholder.typicode.com/users/4 "HTTP/1.1 200 OK"
2026-11-07 17:59:32,572 - INFO - HTTP Request: GET https://jsonplaceholder.typicode.com/users/7 "HTTP/1.1 200 OK"
2026-11-07 17:59:32,575 - INFO - HTTP Request: GET https://jsonplaceholder.typicode.com/users/2 "HTTP/1.1 200 OK"
2026-11-07 17:59:32,577 - INFO - HTTP Request: GET https://jsonplaceholder.typicode.com/users/6 "HTTP/1.1 200 OK"
2026-11-07 17:59:32,581 - INFO - HTTP Request: GET https://jsonplaceholder.typicode.com/users/5 "HTTP/1.1 200 OK"
2026-11-07 17:59:32,585 - INFO - HTTP Request: GET https://jsonplaceholder.typicode.com/users/3 "HTTP/1.1 200 OK"
2026-11-07 17:59:32,587 - INFO - HTTP Request: GET https://jsonplaceholder.typicode.com/users/9 "HTTP/1.1 200 OK"
2026-11-07 17:59:32,589 - INFO 

   user_id  amount transaction_date     status         user_name
0      1.0  250.50        1 03 2025  completed     Leanne Graham
1      2.0    0.00        2 03 2025    pending      Ervin Howell
2      1.0  175.75        1 03 2025  completed     Leanne Graham
3      3.0   89.99        3 03 2025  completed  Clementine Bauch
4      2.0  -10.00        2 03 2025     failed      Ervin Howell


## **Calling Async Function**

In [ ]:
# This will not work
def pipeline():
    users_df = fetch_data_in_async_helper(transactions_df, user_ids)
    print(users_df)

pipeline()

<coroutine object fetch_data_in_async_helper at 0x7d5886693c40>


/tmp/ipykernel_1148/3895440760.py:6: RuntimeWarning: coroutine 'fetch_data_in_async_helper' was never awaited
  pipeline()


In [ ]:
# In a notebook this will generate an error
import asyncio

def pipeline():
    users_df = asyncio.run(
        fetch_data_in_async_helper(transactions_df, user_ids)
    )
    print(user_df)

In [ ]:
# In notebook use this instead.
users_df = await fetch_data_in_async_helper(transactions_df, user_ids)

2026-11-07 17:59:32,629 - INFO - Starting async fetch...
2026-11-07 17:59:32,725 - INFO - HTTP Request: GET https://jsonplaceholder.typicode.com/users/4 "HTTP/1.1 200 OK"
2026-11-07 17:59:32,745 - INFO - HTTP Request: GET https://jsonplaceholder.typicode.com/users/3 "HTTP/1.1 200 OK"
2026-11-07 17:59:32,757 - INFO - HTTP Request: GET https://jsonplaceholder.typicode.com/users/7 "HTTP/1.1 200 OK"
2026-11-07 17:59:32,763 - INFO - HTTP Request: GET https://jsonplaceholder.typicode.com/users/2 "HTTP/1.1 200 OK"
2026-11-07 17:59:32,766 - INFO - HTTP Request: GET https://jsonplaceholder.typicode.com/users/5 "HTTP/1.1 200 OK"
2026-11-07 17:59:32,772 - INFO - HTTP Request: GET https://jsonplaceholder.typicode.com/users/1 "HTTP/1.1 200 OK"
2026-11-07 17:59:32,774 - INFO - HTTP Request: GET https://jsonplaceholder.typicode.com/users/6 "HTTP/1.1 200 OK"
2026-11-07 17:59:32,777 - INFO - HTTP Request: GET https://jsonplaceholder.typicode.com/users/9 "HTTP/1.1 200 OK"
2026-11-07 17:59:32,779 - INFO 